In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-stage-3-2026")

print("Path to dataset files:", path)

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import datasets
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
import torch.optim as optim
import torch.nn.functional as F
from sklearn.metrics import confusion_matrix
import glob
from PIL import Image
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
import seaborn as sns
import pandas as pd
import torchvision.models as models
from sklearn.metrics.pairwise import cosine_similarity
from torchvision.datasets import ImageFolder


In [ ]:
# Write your code here
mean=[0.485, 0.456, 0.406]
std=[0.229, 0.224, 0.225]

transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])


transform_test = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])

# Initialize dataset for Train
train_path = os.path.join(path, "PlantVillage", "train")
test_path = os.path.join(path, "PlantVillage", "test")

train_dataset = ImageFolder(train_path, transform=transform)                                                    ## Replaced SkinCancerDataset with ImageFolder
test_dataset = ImageFolder(test_path, transform=transform_test)                                           ## Replaced SkinCancerDataset with ImageFolder

# Create DataLoader
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

# Get a batch of training images
images, labels = next(iter(train_loader))
print(f"Batch shape: {images.shape}, Labels: {labels}")


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
# Write your code here
class SimplePotatoCNN(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.features = nn.Sequential(

            nn.Conv2d(3, 32, kernel_size=3, padding=1),  # [3,32,32,32]
            nn.BatchNorm2d(32), #Optimization
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),  # [32,64,16,16]
            nn.BatchNorm2d(64), #Optimization
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),   # [64,128,16,16]
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.2),

            nn.Conv2d(256, 512, kernel_size=3, padding=1),   #[256, 512, 4, 4]
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.2),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x



In [ ]:
# Write your code here
def accuracy_from_logits(logits, labels):
    preds = torch.argmax(logits, dim= 1)
    return (preds == labels).float().mean().item()

def train_one_epoch(model, loader, optimizer, criterion):
  model.train().to(device)
  total_loss, total_acc = 0.0, 0.0


  for images, labels in loader:
      images = images.to(device)
      labels = labels.to(device)

      optimizer.zero_grad()

      logits = model(images)
      loss = criterion(logits, labels)

      loss.backward()

      optimizer.step()

      total_loss += loss.item()
      total_acc += accuracy_from_logits(logits.detach(), labels)

  return total_loss / len(loader), total_acc / len(loader)


def evaluate(model, loader, criterion):
    model.eval()
    total_loss, total_acc = 0.0, 0.0

    with torch.no_grad():
        for images, labels in loader:
          images = images.to(device)
          labels = labels.to(device)

          logits = model(images)

          loss = criterion(logits, labels)

          total_loss += loss.item()
          total_acc += accuracy_from_logits(logits, labels)

    return total_loss / len(loader), total_acc / len(loader)




In [ ]:
# Write your code here
# Training setup
model = SimplePotatoCNN().to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(model.parameters(), lr= 0.0005, weight_decay= 0.0001)

num_epochs = 5

#Scheduler for optimization
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=2, min_lr=1e-5
)

# Initialize history tracking
history = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}

# Training loop
for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
    test_loss, test_acc = evaluate(model, test_loader, criterion)

    #Scheduler for optimization
    scheduler.step(float(test_acc))
    current_lr = optimizer.param_groups[0]["lr"]

    # Store history
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["test_loss"].append(test_loss)
    history["test_acc"].append(test_acc)

    print(f'Epoch {epoch+1}/{num_epochs} - Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}, LR: {current_lr:.6f}')


#Plotting
plt.figure(figsize=(15, 5))

plt.subplot(1, 2, 1)
plt.plot(range(1, num_epochs+1), history["train_loss"] , label="train_loss", marker='o')
plt.plot(range(1, num_epochs+1), history["test_loss"], label= "test_loss" , marker='o')
plt.xlabel("epoch")
plt.ylabel("loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(range(1, num_epochs+1), history["train_acc"], label="train_acc", marker='o')
plt.plot(range(1, num_epochs+1), history["test_acc"] , label="test_acc", marker='o')
plt.xlabel("epoch")
plt.ylabel("accuracy")
plt.legend()

plt.show()



In [ ]:
# Write your code here
class SimplePotatoCNN(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.enc1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),  # [3,32,32,32]
            nn.BatchNorm2d(32), #Optimization
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.2),
        )
        self.enc2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),  # [32,64,16,16]
            nn.BatchNorm2d(64), #Optimization
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.2),
        )
        self.enc3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),   # [64,128,8,8]
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.2),
        )
        self.enc4 = nn.Sequential(
            nn.Conv2d(160, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.2),
        )
        self.enc5 = nn.Sequential(
            nn.Conv2d(256, 512, kernel_size=3, padding=1),   #[256, 512, 2, 2]
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.2),
        )
        ###

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        enc1 = self.enc1(x)
        enc2 = self.enc2(enc1)
        enc3 = self.enc3(enc2)
        enc4 = self.enc4(enc3)
        x = torch.cat([self.pool(enc2), enc4], dim=1)
        enc5 = self.enc5(self.pool(x))

        x = self.classifier(enc5)
        return x



In [ ]:
# Write your code here
def accuracy_from_logits(logits, labels):
    preds = torch.argmax(logits, dim= 1)
    return (preds == labels).float().mean().item()

def train_one_epoch(model, loader, optimizer, criterion):
  model.train().to(device)
  total_loss, total_acc = 0.0, 0.0


  for images, labels in loader:
      images = images.to(device)
      labels = labels.to(device)

      optimizer.zero_grad()

      logits = model(images)
      loss = criterion(logits, labels)

      loss.backward()

      optimizer.step()

      total_loss += loss.item()
      total_acc += accuracy_from_logits(logits.detach(), labels)

  return total_loss / len(loader), total_acc / len(loader)


def evaluate(model, loader, criterion):
    model.eval()
    total_loss, total_acc = 0.0, 0.0

    with torch.no_grad():
        for images, labels in loader:
          images = images.to(device)
          labels = labels.to(device)

          logits = model(images)

          loss = criterion(logits, labels)

          total_loss += loss.item()
          total_acc += accuracy_from_logits(logits, labels)

    return total_loss / len(loader), total_acc / len(loader)




In [ ]:
#Retraining
# Write your code here
# Training setup
model = SimplePotatoCNN().to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(model.parameters(), lr= 0.0005, weight_decay= 0.0001)

num_epochs = 5

#Scheduler for optimization
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=2, min_lr=1e-5
)

# Initialize history tracking
history = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}

# Training loop
for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
    test_loss, test_acc = evaluate(model, test_loader, criterion)

    #Scheduler for optimization
    scheduler.step(float(test_acc))
    current_lr = optimizer.param_groups[0]["lr"]

    # Store history
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["test_loss"].append(test_loss)
    history["test_acc"].append(test_acc)

    print(f'Epoch {epoch+1}/{num_epochs} - Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}, LR: {current_lr:.6f}')


#Plotting
plt.figure(figsize=(15, 5))

plt.subplot(1, 2, 1)
plt.plot(range(1, num_epochs+1), history["train_loss"] , label="train_loss", marker='o')
plt.plot(range(1, num_epochs+1), history["test_loss"], label= "test_loss" , marker='o')
plt.xlabel("epoch")
plt.ylabel("loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(range(1, num_epochs+1), history["train_acc"], label="train_acc", marker='o')
plt.plot(range(1, num_epochs+1), history["test_acc"] , label="test_acc", marker='o')
plt.xlabel("epoch")
plt.ylabel("accuracy")
plt.legend()

plt.show()

